# Credit ratios from a statement model

Compute **leverage**, **coverage**, and **liquidity** from evaluated nodes, then layer `run_corporate_analysis`, `pl_summary_report(...).to_text()`, and `credit_assessment_report_text` when available.

## Concept

Credit workpapers pair **P&L** (EBITDA, interest) with **balance sheet** (debt, cash). Statement nodes compute facility-defined metrics, while a typed `CovenantEngine` owns threshold tests, pass/fail decisions, and headroom.

In [ ]:
from finstack_quant.covenants import CovenantEngine, CovenantTemplate
from finstack_quant.core.money import Money
from finstack_quant.statements import Evaluator, ModelBuilder
from finstack_quant.statements_analytics import (
    credit_assessment,
    credit_assessment_report_text,
    pl_summary_report,
    run_corporate_analysis,
)

PERIODS = ["2025Q1", "2025Q2", "2025Q3", "2025Q4"]

def money_series(values):
    return [(period, Money(value, "USD")) for period, value in zip(PERIODS, values, strict=True)]

b = ModelBuilder("credit-demo")
b.periods("2025Q1..Q4", "2025Q2")
b.currency("USD")
b.value_money("revenue", money_series([200.0, 210.0, 218.0, 225.0]))
b.compute("ebitda", "revenue * 0.18")
b.value_money("interest_expense", money_series([7.0, 7.2, 7.5, 7.8]))
b.compute("ebit", "ebitda - interest_expense")
b.value_money("principal_paid", money_series([5.0, 5.0, 5.5, 5.5]))
b.value_money("total_debt", money_series([120.0, 118.0, 115.0, 112.0]))
b.value_money("cash", money_series([15.0, 16.0, 17.0, 18.0]))
b.compute("net_debt", "total_debt - cash")
b.value_money("capex", money_series([8.0, 8.5, 9.0, 9.0]))
b.compute("ufcf", "ebitda - capex")
b.compute("free_cash_flow", "ebitda - interest_expense - principal_paid - capex")
b.compute("debt_to_ebitda", "total_debt / ebitda")
b.compute("interest_coverage", "ebitda / interest_expense")
b.compute("fixed_charge_coverage", "(ebitda - capex) / (interest_expense + principal_paid)")

spec = b.build()
res = Evaluator().evaluate(spec)
eval_json = res.to_json()
model_json = spec.to_json()

as_of = "2025Q4"
engine = CovenantEngine.from_template(CovenantTemplate.lbo_standard(
    initial_leverage=3.0,
    interest_coverage=3.0,
    fixed_charge_coverage=1.5,
    max_capex=12.0,
))
metrics = {
    "debt_to_ebitda": res.get("debt_to_ebitda", as_of),
    "interest_coverage": res.get("interest_coverage", as_of),
    "fixed_charge_coverage": res.get("fixed_charge_coverage", as_of),
    "capex": res.get("capex", as_of),
}
covenant_report = engine.evaluate(metrics, "2025-12-31")
for covenant_id, result in covenant_report.items():
    print(covenant_id, "passed=", result.passed, "headroom in metric units=", result.headroom_raw)

print(pl_summary_report(eval_json, ["revenue", "ebitda", "interest_expense"], PERIODS).to_text())
print(credit_assessment_report_text(eval_json, as_of))

In [ ]:
print("Corporate analysis + downside covenant snapshot")
tv = {"type": "gordon_growth", "stable_growth_rate": 0.02}
analysis_checks = {
    "name": "corporate analysis gate",
    "builtin_checks": [{"type": "non_finite", "nodes": []}],
    "formula_checks": [],
}
corp = run_corporate_analysis(
    spec,
    {
        "wacc": 0.10,
        "terminal_value": tv,
        "net_debt_override": 95.0,
        "cfads_node": "ufcf",
        "interest_coverage_node": "ebitda",
        "check_suite": analysis_checks,
    },
)
print("Corporate analysis:", type(corp).__name__)
print("statement nodes:", corp.statement.node_count)
if corp.equity is not None:
    print("Equity value:", corp.equity.equity_value)

ca = credit_assessment(res, "2025Q4")
print("credit_assessment:", ca.period, "leverage=", ca.leverage_ratio, "coverage=", ca.interest_coverage)

downside_metrics = {
    "debt_to_ebitda": 3.4,
    "interest_coverage": 2.6,
    "fixed_charge_coverage": 1.3,
    "capex": 9.0,
}
downside_report = engine.evaluate(downside_metrics, "2025-12-31")
print("Downside covenant results:")
for covenant_id, result in downside_report.items():
    print(covenant_id, "passed=", result.passed, "headroom in metric units=", result.headroom_raw)

## Coverage on the contractual borrowing schedule

Use the identified capital cashflow run for the selected borrowing. Cash DSCR
uses scheduled cash interest and principal due. The collateral valuation has
its own source and closing date, so another quarter with no matching value
remains unavailable.


In [ ]:
from finstack_quant.core.market_data import MarketContext
from finstack_quant.statements_analytics import CreditContextSpec, evaluate_credit_context
from finstack_quant.valuations.instruments import Bond

as_of = "2025-01-01"
bond = Bond.with_convention(
    "issuer-note", Money(100, "USD"), 0.06, "2025-01-01", "2026-01-01", "us_treasury", "USD-OIS"
)
schedule_model = ModelBuilder("scheduled-credit").periods("2025Q1..Q4").currency("USD")
schedule_model.source_snapshot("contractual-issuer-note-2025-01")
schedule_model.value_money("cfads", [(period, Money(10, "USD")) for period in PERIODS])
schedule_model.add_debt("issuer-note", bond)
dated = Evaluator().evaluate_with_market(schedule_model.build(), MarketContext(), as_of, as_of)
assert dated.provenance is not None
report = evaluate_credit_context(
    dated,
    CreditContextSpec(
        borrowing_id="issuer-note",
        run_id=dated.provenance["run_id"],
        currency="USD",
        periods=["2025Q2", "2025Q3"],
        measures=["cash_dscr", "loan_to_value"],
        cfads_node="cfads",
        reference_values=[{
            "measure": "loan_to_value",
            "date": "2025-06-30",
            "amount": Money(200, "USD"),
            "source_id": "approved-appraisal-2025Q2",
        }],
    ),
)
leverage = [row for row in report.rows if row.measure == "loan_to_value"]
assert leverage[0].ratio == 0.5
assert leverage[0].reference_source_id == "approved-appraisal-2025Q2"
assert leverage[1].ratio is None and leverage[1].reason
display(report.to_pandas())


## Takeaways

- **Leverage and coverage** are statement nodes aligned to facility definitions.
- `credit_assessment_report_text` is a fast text view of the evaluated statements.
- `CovenantEngine.evaluate` is the sole source for covenant pass/fail and headroom in both base and downside snapshots.